# Organization for the next version of the validation dataset
* Images are downloaded by running the snakemake workflow with the "validation-cases" configuration file
* Image labels follow the format `NNN-region_name-XXXkm-YYYYMMDD.satellite_name.image_type.250m.tiff` where NNN is the case number and XXX is the horizontal scale of the image.
* 100 km images are subsets of the 250 km subsets, which are subsets of the 500 km images. If the 100 km image center is far from the image boundary, then the other images are centered on the same location, otherwise the images are shifted to fit within the boundary.

This notebook has code to pull results from the workflow output into the repository. Because the case names in the workflow do not have the spatial scale, running this notebook without being careful can overwrite existing data. 

## TODO ITEMS
* Script to copy truecolor and falsecolor images
* Script to copy landmask images
* Script to extract the SOIT passtime estimates
* Script to copy the cloud fraction images
* Script to extract and interpolate the UBremen sea ice concentration at the sample times and locations
* Script to generate cloud / ice / water samples using the MODIS clouds, UBremen ice, and the 500 km images. (plus, discuss methodology)

In [2]:
import shutil as sh
import os
import pandas as pd

In [9]:
region_df = {region: data for region, data in df.groupby("region_name")}
region_df["beaufort_sea"]

,case_number,region_name,center_lon,center_lat,center_x,center_y,start,end,scale,center_crs,region,target_crs,pipeline,target_width,target_height
42,43,beaufort_sea,-152.729,79.043,-1133879,362500,2019-08-13,2019-08-13,250,EPSG:4326,beaufort_sea_043,EPSG:3413,FSPipeline,500000,500000
43,44,beaufort_sea,-150.180,74.665,-1612500,437500,2020-08-08,2020-08-08,250,EPSG:4326,beaufort_sea_044,EPSG:3413,FSPipeline,500000,500000
44,45,beaufort_sea,-148.079,78.311,-1237500,287500,2017-09-25,2017-09-25,250,EPSG:4326,beaufort_sea_045,EPSG:3413,FSPipeline,500000,500000
45,46,beaufort_sea,-142.673,70.312,-2133879,287500,2020-07-08,2020-07-08,250,EPSG:4326,beaufort_sea_046,EPSG:3413,FSPipeline,500000,500000
46,47,beaufort_sea,-141.351,70.367,-2133879,237500,2022-03-02,2022-03-02,250,EPSG:4326,beaufort_sea_047,EPSG:3413,FSPipeline,500000,500000
47,48,beaufort_sea,-140.687,70.390,-2133879,212500,2021-04-27,2021-04-27,250,EPSG:4326,beaufort_sea_048,EPSG:3413,FSPipeline,500000,500000
48,49,beaufort_sea,-140.687,70.390,-2133879,212500,2016-03-05,2016-03-05,250,EPSG:4326,beaufort_sea_049,EPSG:3413,FSPipeline,500000,500000
49,50,beaufort_sea,-139.733,71.966,-1962500,162500,2007-09-02,2007-09-02,250,EPSG:4326,beaufort_sea_050,EPSG:3413,FSPipeline,500000,500000
50,51,beaufort_sea,-139.554,76.974,-1412500,112500,2010-09-25,2010-09-25,250,EPSG:4326,beaufort_sea_051,EPSG:3413,FSPipeline,500000,500000
51,52,beaufort_sea,-137.372,70.661,-2112500,87500,2021-06-24,2021-06-24,250,EPSG:4326,beaufort_sea_052,EPSG:3413,FSPipeline,500000,500000


In [102]:
df = pd.read_csv('../data/metadata/validation-region-case-500km.csv')

workflow_loc = "/Users/dwatkin2/Documents/research/packages/IceFloeTracker.jl/workflow/results/filetype/"
save_loc = "../data/modis/"
os.listdir(workflow_loc)

# Already done: 
# Beaufort Sea, Bering Chukchi Seas, East Siberian Sea

region = 'baffin_bay'
dx = 500
for imtype in ['truecolor', 'falsecolor', 'landmask']:
    for f in os.listdir(os.path.join(workflow_loc, imtype)):
        if region in f:
            region_case, px_scale, date, satellite, ftype = f.split('.')
            case_number = region_case.split('_')[-1]
    
            fname = '{cn}-{rg}-{dx}km-{date}.{satellite}.{imtype}.{scale}.tiff'.format(
                cn=case_number,
                rg=region,
                dx=dx,
                imtype=imtype,
                date=date.replace('-', ''),
                satellite=satellite,
                scale=px_scale
            )
            
            sh.copy2(os.path.join(workflow_loc, imtype, f), os.path.join(save_loc, imtype, fname))
imtype = 'cloudfraction'
for f in os.listdir(os.path.join(workflow_loc, 'cloud')):
    if region in f:
        region_case, px_scale, date, satellite, ftype = f.split('.')
        case_number = region_case.split('_')[-1]

        fname = '{cn}-{rg}-{dx}km-{date}.{satellite}.{imtype}.{scale}.tiff'.format(
            cn=case_number,
            rg=region,
            dx=dx,
            imtype=imtype,
            date=date.replace('-', ''),
            satellite=satellite,
            scale=px_scale
        )
        
        sh.copy2(os.path.join(workflow_loc, 'cloud', f), os.path.join(save_loc, imtype, fname))

Check existence for all expected files

In [103]:
missing = {}
imtype = 'falsecolor'
for region in region_df:
    missing[region] = []
    for row, data in region_df[region].iterrows():
        for satellite in ['aqua', 'terra']:
            fname = '{cn}-{rg}-{dx}km-{date}.{satellite}.{imtype}.{scale}m.tiff'.format(
                cn=str(data.case_number).zfill(3),
                rg=region,
                dx=500,
                imtype=imtype,
                date=data.start.replace('-', ''),
                satellite=satellite,
                scale=250
            )
            if not os.path.exists(os.path.join(save_loc, imtype, fname)):
                missing[region].append(fname)
for region in missing:
    print(region, len(missing[region]))

baffin_bay 0
barents_kara_seas 1
beaufort_sea 0
bering_chukchi_seas 0
east_siberian_sea 0
greenland_sea 0
hudson_bay 0
laptev_sea 0
sea_of_okhostk 0


In [104]:
missing = {}
imtype = 'truecolor'
for region in region_df:
    missing[region] = []
    for row, data in region_df[region].iterrows():
        for satellite in ['aqua', 'terra']:
            fname = '{cn}-{rg}-{dx}km-{date}.{satellite}.{imtype}.{scale}m.tiff'.format(
                cn=str(data.case_number).zfill(3),
                rg=region,
                dx=500,
                imtype=imtype,
                date=data.start.replace('-', ''),
                satellite=satellite,
                scale=250
            )
            if not os.path.exists(os.path.join(save_loc, imtype, fname)):
                missing[region].append(fname)
for region in missing:
    print(region, len(missing[region]))

baffin_bay 0
barents_kara_seas 1
beaufort_sea 0
bering_chukchi_seas 0
east_siberian_sea 0
greenland_sea 0
hudson_bay 0
laptev_sea 0
sea_of_okhostk 0


In [100]:
missing = {}
imtype = 'landmask'
for region in region_df:
    missing[region] = []
    for row, data in region_df[region].iterrows():
        for satellite in ['aqua', 'terra']:
            fname = '{cn}-{rg}-{dx}km-{date}.{satellite}.{imtype}.{scale}m.tiff'.format(
                cn=str(data.case_number).zfill(3),
                rg=region,
                dx=500,
                imtype=imtype,
                date=data.start.replace('-', ''),
                satellite=satellite,
                scale=250
            )
            if not os.path.exists(os.path.join(save_loc, imtype, fname)):
                missing[region].append(fname)
for region in missing:
    print(region, len(missing[region]))

baffin_bay 11
barents_kara_seas 9
beaufort_sea 0
bering_chukchi_seas 0
east_siberian_sea 0
greenland_sea 0
hudson_bay 0
laptev_sea 0
sea_of_okhostk 0


In [106]:
missing = {}
imtype = 'cloudfraction'
for region in region_df:
    missing[region] = []
    for row, data in region_df[region].iterrows():
        for satellite in ['aqua', 'terra']:
            fname = '{cn}-{rg}-{dx}km-{date}.{satellite}.{imtype}.{scale}m.tiff'.format(
                cn=str(data.case_number).zfill(3),
                rg=region,
                dx=500,
                imtype=imtype,
                date=data.start.replace('-', ''),
                satellite=satellite,
                scale=250
            )
            if not os.path.exists(os.path.join(save_loc, imtype, fname)):
                missing[region].append(fname)
for region in missing:
    print(region, len(missing[region]))

baffin_bay 1
barents_kara_seas 1
beaufort_sea 0
bering_chukchi_seas 0
east_siberian_sea 0
greenland_sea 0
hudson_bay 0
laptev_sea 0
sea_of_okhostk 0


# TODO: SOIT
Loop through the workflow directory, read the overpass.txt, and append to a list with the case numbers.

# TODO: IFT output
Loop through the workflow directory, load the segmentation.nc file, then write it to a new location with compression.

As a temporary measure, flip the images to correct for the error in the ArchiveV1 module.